# 16. Random forests and bagging

**Level:** Intermediate

**Prerequisites:** 15: decision trees.

[Course index](00_START_HERE.ipynb) | [Previous](15_Decision_Trees.ipynb) | [Next](17_Gradient_Boosting.ipynb)

**Topics:** Random forests, Bagging, Bootstrap, Variance reduction, Feature importance.

**Matching script:** `examples\16_Random_Forests.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A random forest combines many trees trained with randomized data samples and feature choices.
Averaging regression predictions or combining classification votes can reduce variance compared
with one unstable tree. The benefit depends on individual trees being useful and not perfectly
correlated.

n_estimators controls how many trees are built; depth and leaf-size settings control tree
complexity. More trees consume memory and inference time. Random forests can be strong tabular
baselines but usually do not extrapolate numeric targets beyond the learned leaf values.

Permutation importance measures how a scoring metric changes when a feature is shuffled on
held-out data. Correlated features can share importance, and scores can be noisy. It measures
model reliance under that perturbation, not causality.

## Mathematics or data contract

Bagging averages predictions across randomized fitted models, reducing variance when errors are not perfectly correlated.

## Important limitation

A synthetic importance ranking is not a causal explanation or production feature audit.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: A bounded forest with held-out permutation importance

**Purpose:** Train an ensemble without using all CPU cores.

### Read the code in this order
1. Create a controlled dataset and held-out split.
2. Fit a modest n_jobs=1 forest.
3. Evaluate accuracy and compute permutation importance.
4. Check importance shape and finite values.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
held-out accuracy: 0.907
permutation importance: [0.013, 0.036, 0.169, -0.009, -0.004]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=300, n_features=5, n_informative=3,
                           class_sep=1.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=40, min_samples_leaf=2, n_jobs=1, random_state=42)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
importance = permutation_importance(model, X_test, y_test, n_repeats=3, random_state=42, n_jobs=1)
assert score > 0.75 and importance.importances_mean.shape == (5,)
assert np.isfinite(importance.importances_mean).all()
print("held-out accuracy:", round(score, 3))
print("permutation importance:", np.round(importance.importances_mean, 3).tolist())

held-out accuracy: 0.907
permutation importance: [0.013, 0.036, 0.169, -0.009, -0.004]


### Interpret example 1

The result should demonstrate this purpose: **Train an ensemble without using all CPU cores.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Measure how n_estimators changes fit time and held-out performance without repeatedly selecting on test data.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Random forests` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Bagging averages predictions across randomized fitted models, reducing variance when errors are not perfectly correlated.
- Name the boundary of the demonstration: A synthetic importance ranking is not a causal explanation or production feature audit.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.